# Lab 1 — Version 2

## Approach
Inspired by Lecture 2, “Social Media User Profiling,” this solution combines text representations with author-history and attachment features using early and late fusion.

Version 1 used recent author-history and writing-style features with a TF-IDF classifier. Version 2 adds richer context, SVD/LSA features and frozen multilingual sentence embeddings.

## Data and validation
The official split is preserved:
- Train: 71,395 posts from 3,295 authors.
- Test: 17,623 posts from 824 different authors.

Configuration selection uses three author-disjoint validation splits within train, reserving 20% of authors for validation with seeds 42, 123 and 2026.

TF-IDF, SVD, PCA and classifiers are fitted only on the fitting portion of each split. The text encoder remains frozen. The configuration with the highest mean validation macro-F1 is then refitted on the full training set.

The test set was previously evaluated in Version 1, but Version 2 settings were selected using training-data validation only.

## Features and models

### Author and attachment features
In addition to Version 1 features, the model uses:
- Post length relative to the author's previous posts.
- Recent reaction levels, variability and performance relative to the author's usual.
- How frequently the author uses the current post format.
- Image count, document page count and attachment indicators.

Historical features use earlier posts only. The current post's engagement counters and label are excluded from model inputs.

### Text models
Three models contribute to the final prediction:

1. **TF-IDF classifier:** word unigrams and bigrams with balanced logistic regression.
2. **LSA + tabular model:** TF-IDF compressed to 64 dimensions with SVD, normalized and combined with author/context features in a gradient boosting classifier.
3. **Embeddings + tabular model:** frozen multilingual MiniLM embeddings reduced from 384 to 64 dimensions with PCA and combined with the same context features in a gradient boosting classifier.

The encoder is `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`. Text processing runs locally.

Final prediction weights are **15% TF-IDF, 42.5% LSA/tabular and 42.5% embeddings/tabular**. The typical-class score is multiplied by 1.6 before choosing the predicted class.

## Validation results
Each model family uses its best shared configuration across the three validation splits.

| Model family | Mean macro-F1 |
|---|---:|
| Version 1 | 0.6185 |
| Additional context + TF-IDF classifier | 0.6371 |
| LSA/tabular + TF-IDF classifier | 0.6477 |
| Embeddings/tabular + TF-IDF classifier | 0.6441 |
| Final ensemble | **0.6506** |

The final ensemble improves over Version 1 on all three validation splits. Embeddings alone do not outperform LSA here, but combining the two representations improves the mean score.

## Test results
| Model | Macro-F1 | QWK | Accuracy |
|---|---:|---:|---:|
| Official baseline | 0.5410 | 0.4576 | 0.5465 |
| Version 1 | 0.6052 | 0.5240 | 0.6444 |
| Version 2 | **0.6412** | **0.5699** | **0.6860** |

Version 2 improves macro-F1 by **0.0360** over Version 1. F1 increases for all three classes:

| Class | Version 1 | Version 2 |
|---|---:|---:|
| Below | 0.4850 | 0.5185 |
| Typical | 0.7085 | 0.7487 |
| Above | 0.6220 | 0.6565 |

Final training was reproduced locally using cached embeddings. The official `evaluate.py` verified all 17,623 test predictions.

## Limitations
- Historical engagement and follower counts are snapshots collected at crawl time, so they do not fully reconstruct information available when a post was published.
- The neural encoder reads the first 128 tokens; the TF-IDF component processes the full text.
- Repeated validation splits overlap, and the test set has been used in earlier experiments. These results are not a statistical significance claim.

## Reproduction
Place the dataset in `data/` and run from the repository root:

```bash
python -m pip install -r requirements-v2.txt
python improved_v2.py --stage prepare
python encode_posts_v2.py
python improved_v2.py --stage validate --embeddings results/v2_cache/embeddings.npy
python improved_v2.py --stage final --embeddings results/v2_cache/embeddings.npy
python evaluate.py --predictions lab1_submission_v2/predictions.csv
```

If embeddings and the selected configuration are already available, only the last two commands are needed. Training with cached embeddings requires only the original `requirements.txt`.

Tested with Python 3.14.8 and scikit-learn 1.9.1. Embeddings were generated using sentence-transformers 6.2.0 on Apple MPS.

Final outputs are saved in `lab1_submission_v2/`. Intermediate data and embeddings remain under the Git-ignored `results/` directory.

The notebook cells below display saved results; they do not retrain the models.


In [1]:
import json
from pathlib import Path
results = json.loads(Path("lab1_submission_v2/metrics.json").read_text())
print(json.dumps(results["test"], indent=2))


{
  "macro_f1": 0.6412,
  "qwk": 0.5699,
  "accuracy": 0.686,
  "f1_by_class": [
    0.5184969414506263,
    0.7487003341997772,
    0.6565242832065535
  ],
  "confusion_matrix": [
    [
      1780,
      1719,
      42
    ],
    [
      1467,
      8065,
      959
    ],
    [
      78,
      1269,
      2244
    ]
  ]
}
